# CameraTrackingPose — Development Pipeline

A step-by-step guide through every component of the system.
Use this notebook to **test**, **debug**, and **iterate** on each part independently.

---

## Architecture Overview

```
┌──────────────────────────────────────────────────────────────────┐
│  Input Sources                                                   │
│  Camera 0 ─┐                                                     │
│  Camera 1 ─┤──► CameraManager  (enable/disable per camera)      │
│  Video     ─┘         │                                          │
│                        ▼                                          │
│              PoseEstimator (RTMPose whole-body, 133 kps)         │
│                        │                                          │
│              ┌─────────┴──────────┐                              │
│              ▼                    ▼                              │
│         DisplayManager        VRMMapper                          │
│      (OpenCV grid + skeleton)  (kps → quaternions)              │
│              │                    │                              │
│         Preview window      WebSocket server                     │
│         (debug / dev)        → Unity / Godot                    │
└──────────────────────────────────────────────────────────────────┘
```

### Key design decisions
| Decision | Choice | Reason |
|---|---|---|
| Pose model | RTMPose whole-body (MMPose 1.x) | Real-time, 133 kps (body+face+hands) |
| Closest-person heuristic | Largest bounding box | Cheapest proxy for depth |
| VRM bone rotations | World-space quaternions via scipy | Easy to consume in Unity |
| Unity communication | WebSocket (JSON) | Low-latency, simple client-side |
| Camera disable | Per-index toggle (runtime + CLI) | Flexible rig management |

---
## 1. Environment Check

In [ ]:
import sys, os

# Make sure src/ is importable from this notebook
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Python : {sys.version}")
print(f"Root   : {ROOT}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}  ",
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

import cv2
print(f"OpenCV : {cv2.__version__}")

import mmpose
print(f"MMPose : {mmpose.__version__}")

---
## 2. Configuration

Edit `Config` here to change cameras, device, ports, etc.

In [ ]:
from src.config import Config

cfg = Config(
    camera_indices   = [0, 1],   # cameras to open
    disabled_cameras = [1],      # start with camera 1 disabled
    video_path       = None,     # set to a file path to use video instead
    video_loop       = True,
    device           = "cuda:0",
    ws_port          = 8765,
    target_fps       = 30,
    skip_frames      = 0,        # 0 = run inference every frame
    kp_confidence    = 0.3,
)

print("Config OK")
print(f"  cameras        : {cfg.camera_indices}")
print(f"  disabled start : {cfg.disabled_cameras}")
print(f"  device         : {cfg.device}")
print(f"  ws_port        : {cfg.ws_port}")

---
## 3. Camera Manager

### 3-A  Open cameras and inspect states

In [ ]:
from src.camera_manager import CameraManager

cam_mgr = CameraManager(cfg)
cam_mgr.open_cameras()

print("Camera states after open():")
for idx, enabled in cam_mgr.get_states().items():
    print(f"  Camera {idx} → {'ENABLED' if enabled else 'DISABLED'}") 

### 3-B  Enable / Disable cameras

You can call these at **any time** — even during a running session.

In [ ]:
# Enable camera 1
cam_mgr.enable_camera(1)

# Disable camera 0
cam_mgr.disable_camera(0)

# Toggle camera 0 back on
new_state = cam_mgr.toggle_camera(0)
print(f"Camera 0 toggled → {'ENABLED' if new_state else 'DISABLED'}")

print("\nFinal states:")
for idx, enabled in cam_mgr.get_states().items():
    print(f"  Camera {idx} → {'ENABLED ✓' if enabled else 'DISABLED ✗'}")

### 3-C  Grab and display a single frame

In [ ]:
import matplotlib.pyplot as plt

frames = cam_mgr.read_frames()
print(f"Active frames received: {list(frames.keys())}")

fig, axes = plt.subplots(1, max(1, len(frames)), figsize=(14, 5))
if len(frames) == 1:
    axes = [axes]

for ax, (cam_idx, frame) in zip(axes, frames.items()):
    ax.imshow(frame[:, :, ::-1])   # BGR → RGB
    ax.set_title(f"Camera {cam_idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### 3-D  Load a video file instead of live cameras

Uncomment and change the path to test with a video.

In [ ]:
# VIDEO_PATH = r"C:\path\to\your\video.mp4"
# cam_mgr.load_video(VIDEO_PATH)
# frames = cam_mgr.read_frames()
# print(f"Video frame shape: {next(iter(frames.values())).shape}")

---
## 4. Pose Estimation (RTMPose Whole-Body)

First run will **download the model weights** (~100 MB).  
Subsequent runs load from the MMPose cache.

### 4-A  Load the estimator

In [ ]:
from src.pose_estimator import PoseEstimator

estimator = PoseEstimator(cfg)
print("Estimator ready.")

### 4-B  Run inference on a captured frame

In [ ]:
import numpy as np

frames = cam_mgr.read_frames()
test_cam = next(iter(frames))              # pick the first active camera
test_frame = frames[test_cam]

results = estimator.estimate(test_frame)
closest = PoseEstimator.get_closest_person(results)

print(f"Detected {len(results)} person(s) in Camera {test_cam}")
if closest:
    print(f"Closest person bounding box : {closest.bbox[:4].astype(int)}")
    print(f"Keypoint shape              : {closest.keypoints.shape}")
    print(f"Mean keypoint confidence    : {closest.keypoint_scores.mean():.3f}")
    body_kps = closest.keypoints[:17]
    print(f"\nBody keypoints (0-16):")
    kp_names = ["nose","l_eye","r_eye","l_ear","r_ear",
                "l_sh","r_sh","l_elbow","r_elbow","l_wrist","r_wrist",
                "l_hip","r_hip","l_knee","r_knee","l_ank","r_ank"]
    for name, kp, sc in zip(kp_names, body_kps, closest.keypoint_scores[:17]):
        print(f"  {name:8s}  x={kp[0]:6.1f}  y={kp[1]:6.1f}  score={sc:.2f}")

### 4-C  Visualize skeleton overlay

In [ ]:
from src.display import DisplayManager

display = DisplayManager(cfg)

if closest:
    annotated = display.draw_pose(test_frame, closest, highlight=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated[:, :, ::-1])
    plt.title(f"Camera {test_cam} — closest person highlighted")
    plt.axis("off")
    plt.show()
else:
    print("No person detected in this frame.")

---
## 5. VRM Bone Mapping

Maps 133 whole-body keypoints → VRM `HumanBodyBones` as quaternions `[x, y, z, w]`.

**Reference model**: `Model/Minami.vrm` (VRM 0.x humanoid skeleton spec)

### 5-A  Run the mapper

In [ ]:
from src.vrm_mapper import VRMMapper

vrm = VRMMapper(confidence=cfg.kp_confidence)

if closest:
    bones = vrm.map(closest)
    print(f"Mapped {len(bones)} VRM bones:\n")
    for bone, quat in sorted(bones.items()):
        q = [f"{v:+.4f}" for v in quat]
        print(f"  {bone:<22s}  [{', '.join(q)}]")
else:
    print("No pose result available — run Section 4-B first.")

### 5-B  Bone coverage overview

In [ ]:
VRM_ALL_BONES = [
    # Spine chain
    "Hips", "Spine", "Chest", "UpperChest", "Neck", "Head",
    # Left arm
    "LeftShoulder", "LeftUpperArm", "LeftLowerArm", "LeftHand",
    # Right arm
    "RightShoulder", "RightUpperArm", "RightLowerArm", "RightHand",
    # Left leg
    "LeftUpperLeg", "LeftLowerLeg", "LeftFoot", "LeftToes",
    # Right leg
    "RightUpperLeg", "RightLowerLeg", "RightFoot", "RightToes",
]

if closest:
    covered   = [b for b in VRM_ALL_BONES if b in bones]
    uncovered = [b for b in VRM_ALL_BONES if b not in bones]
    print(f"Coverage: {len(covered)}/{len(VRM_ALL_BONES)} required VRM bones\n")
    print("Covered  :", covered)
    print("Missing  :", uncovered)
    print("\nExtra (fingers / face):",
          [b for b in bones if b not in VRM_ALL_BONES][:10], "...")

---
## 6. WebSocket Server

Start the server and test that it broadcasts correctly.

### 6-A  Start the server

In [ ]:
from src.websocket_server import PoseWebSocketServer

ws_server = PoseWebSocketServer(cfg)
ws_server.start()

print(f"WebSocket server started on ws://localhost:{cfg.ws_port}")
print("Connect from Unity using:  ws://127.0.0.1:8765")

### 6-B  Send a test pose message

You can test the connection from Unity or from any WebSocket client tool.

In [ ]:
import time

# Send the last mapped bones (if available) or a synthetic identity pose
test_bones = bones if closest else {
    "Hips":         [0.0, 0.0, 0.0, 1.0],
    "Spine":        [0.0, 0.0, 0.0, 1.0],
    "LeftUpperArm": [0.0, 0.0, 0.0, 1.0],
    "RightUpperArm":[0.0, 0.0, 0.0, 1.0],
}

ws_server.send_pose(camera_id=0, bones=test_bones)
print(f"Message sent — {len(test_bones)} bones")
print(f"Connected clients: {ws_server.connected_clients}")

### 6-C  WebSocket payload format

Every frame, the server broadcasts:

```json
{
  "camera": 0,
  "bones": {
    "Hips":         [x, y, z, w],
    "Spine":        [x, y, z, w],
    "LeftUpperArm": [x, y, z, w],
    ...
  },
  "keypoints": [[x, y], [x, y], ...]  // 133 keypoints
}
```

**Unity C# consumer example:**

```csharp
// Parse the quaternion for Hips
float[] q = data["bones"]["Hips"];
animator.SetBoneLocalRotation(
    HumanBodyBones.Hips,
    new Quaternion(q[0], q[1], q[2], q[3])
);
```

---
## 7. Multi-Camera Grid Display Test

In [ ]:
frames = cam_mgr.read_frames()

all_poses  = {}
closest_p  = {}

for cam_idx, frame in frames.items():
    preds = estimator.estimate(frame)
    all_poses[cam_idx]  = preds
    closest_p[cam_idx]  = PoseEstimator.get_closest_person(preds)

grid = display.build_grid(
    frames,
    all_poses=all_poses,
    closest=closest_p,
    labels=cam_mgr.get_labels(),
)

plt.figure(figsize=(14, 7))
plt.imshow(grid[:, :, ::-1])
plt.title("Multi-camera grid preview")
plt.axis("off")
plt.show()
print(f"Grid shape: {grid.shape}  |  Active cameras: {cam_mgr.active_indices}")

---
## 8. Run the Application

There are **two ways** to run the full pipeline:

---

### Option A — GUI Application (recommended)

A full tkinter window with live camera feed embedded, camera toggles, video loader, settings panel and WebSocket status.

**Step-by-step:**

1. **Set up the environment** (first time only)  
   Double-click `setup_venv.bat` — creates the venv and installs PyTorch+CUDA + all deps.

2. **Launch the GUI**  
   Double-click `run_ui.bat`, or from a terminal with venv active:
   ```
   venv\Scripts\activate
   python -m src.ui_app
   ```

3. **In the window:**
   - **Cameras panel** — checkboxes to enable/disable each camera live; rename labels; add/remove cameras by index
   - **Video File panel** — Browse to load `.mp4`/`.avi`/etc.; toggle Loop on/off
   - **Settings panel** — device (`cuda:0` / `cpu`), FPS, skip-frames, confidence
   - **WebSocket panel** — set port (default 8765); shows live client count when Unity connects
   - **Start** — loads RTMPose model (~5-15 s first run) then starts live feed
   - **Stop** — gracefully stops the pipeline

4. **Connect Unity** to `ws://127.0.0.1:8765` — the WebSocket status turns green.

---

### Option B — CLI (headless / scripted)

```bash
venv\Scripts\activate

# Two cameras, camera 1 starts disabled
python -m src.main --cameras 0 1 --disable 1

# Video file
python -m src.main --video path\to\video.mp4

# CPU only
python -m src.main --device cpu
```

**Controls (OpenCV window must have focus):**  
`Q` / `Esc` — quit  |  `0`-`9` — toggle camera on/off

In [ ]:
# Release any open cameras before handing over to App
cam_mgr.release()

from src.main import App

app = App(cfg)
app.setup()
app.run()    # blocks until Q / Esc is pressed

---
## 9. CLI Quick Reference

```bash
# Activate venv first
venv\Scripts\activate

# Two cameras, start with camera 1 disabled
python -m src.main --cameras 0 1 --disable 1

# Video file (loops forever)
python -m src.main --video path\to\video.mp4

# Performance mode: run inference every other frame
python -m src.main --cameras 0 1 --skip 1 --fps 30

# CPU-only (no GPU)
python -m src.main --device cpu
```